# Exercise 08: Membership Inference Attack

**Goal:** Train an attack model that determines whether a data point was in the training set, exploiting the model's overconfidence on training data.

**Time:** ~10 min

## Background

**Membership inference** (Shokri et al. 2017) exploits the fact that ML models are typically more confident on training data than on unseen data.

**Shadow model attack:**
1. Train a *shadow model* on known data, mimicking the target model.
2. Collect confidence vectors from the shadow model on *member* (training) and *non-member* (test) data.
3. Train a binary *attack model* on these confidence vectors.
4. Apply the attack model to the target model's outputs to infer membership.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import helper

torch.manual_seed(42)
np.random.seed(42)

transform = transforms.ToTensor()
full_train = datasets.MNIST('./data', train=True,  download=True, transform=transform)
full_test  = datasets.MNIST('./data', train=False, download=True, transform=transform)

# 1000-sample training sets: small enough to cause strong overfitting (~10% gap).
# Balanced member/non-member splits so the random-guess baseline is truly 50%.
target_train = Subset(full_train, range(0,    1000))
shadow_train = Subset(full_train, range(1000, 2000))
target_test  = Subset(full_test,  range(0,    1000))
shadow_test  = Subset(full_test,  range(1000, 2000))

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(32*7*7,64), nn.ReLU(), nn.Linear(64,10)
        )
    def forward(self, x): return self.net(x)

def train_model(model, dataset, epochs=30):
    loader  = DataLoader(dataset, batch_size=32, shuffle=True)
    opt     = torch.optim.Adam(model.parameters())
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for x, y in loader:
            opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    model.eval()

def evaluate(model, dataset):
    loader = DataLoader(dataset, batch_size=512)
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            correct += (model(x).argmax(1) == y).sum().item(); total += len(y)
    return correct / total

def get_confidence_vectors(model, dataset):
    """Full 10-dim softmax — used for visualization."""
    loader = DataLoader(dataset, batch_size=512)
    confs  = []
    with torch.no_grad():
        for x, _ in loader:
            confs.append(torch.softmax(model(x), dim=1).numpy())
    return np.concatenate(confs)

def get_loss_features(model, dataset):
    """Per-sample [max_confidence, cross_entropy_loss] — the key attack signal.
    Members have loss ≈ 0 (memorized); non-members have loss > 0.
    """
    loader  = DataLoader(dataset, batch_size=512)
    loss_fn = nn.CrossEntropyLoss(reduction='none')
    feats   = []
    with torch.no_grad():
        for x, y in loader:
            logits   = model(x)
            probs    = torch.softmax(logits, dim=1)
            max_conf = probs.max(dim=1).values.numpy()
            loss     = loss_fn(logits, y).numpy()
            feats.append(np.stack([max_conf, loss], axis=1))
    return np.concatenate(feats)

print("Training target model...")
target_model = SmallCNN(); train_model(target_model, target_train)
print(f"  train acc: {evaluate(target_model, target_train):.2%}  test acc: {evaluate(target_model, target_test):.2%}")
print("Training shadow model...")
shadow_model = SmallCNN(); train_model(shadow_model, shadow_train)
print(f"  train acc: {evaluate(shadow_model, shadow_train):.2%}  test acc: {evaluate(shadow_model, shadow_test):.2%}")

shadow_member_feats    = get_loss_features(shadow_model, shadow_train)
shadow_nonmember_feats = get_loss_features(shadow_model, shadow_test)

X_attack = np.concatenate([shadow_member_feats, shadow_nonmember_feats])
y_attack  = np.concatenate([np.ones(len(shadow_member_feats)),
                             np.zeros(len(shadow_nonmember_feats))])
print(f"\nAttack training set: {len(X_attack)} samples")
print(f"  mean loss  — members: {shadow_member_feats[:,1].mean():.4f}  non-members: {shadow_nonmember_feats[:,1].mean():.4f}")

## 🎯 TODO: Train the Attack Model

Use the shadow model's loss features to train a logistic regression classifier, then evaluate it on the target model's outputs.

**Key insight:** Members have loss $\approx 0$ (the model memorized them); non-members have loss $> 0$ even when correctly classified. This loss gap is the attack signal.

In [ ]:
# 🎯 TODO: Train a binary attack model to distinguish members from non-members.
#
# Steps:
# 1. Train an attack model on (X_attack, y_attack):
#      attack_model = LogisticRegression(max_iter=1000)
#      attack_model.fit(X_attack, y_attack)
#
# 2. Build the TARGET model's feature set using get_loss_features:
#      target_member_feats    = get_loss_features(target_model, target_train)
#      target_nonmember_feats = get_loss_features(target_model, target_test)
#      X_target = np.concatenate([target_member_feats, target_nonmember_feats])
#      y_target = np.concatenate([np.ones(len(target_member_feats)),
#                                  np.zeros(len(target_nonmember_feats))])
#
# 3. Predict membership and evaluate:
#      y_pred = attack_model.predict(X_target)
#      y_prob = attack_model.predict_proba(X_target)[:, 1]
#      Print: accuracy, precision, recall, AUC
#      Baseline (random guessing) = 50% accuracy / AUC = 0.5

attack_model           = None   # 🎯 TODO
target_member_feats    = None   # 🎯 TODO
target_nonmember_feats = None   # 🎯 TODO

## Visualize and Discuss

In [ ]:
# Visualize loss and attack score distributions
# (runs only after the TODO above is complete)
if target_member_feats is not None and attack_model is not None:
    mem_scores  = attack_model.predict_proba(target_member_feats)[:, 1]
    nonm_scores = attack_model.predict_proba(target_nonmember_feats)[:, 1]

    helper.plot_distribution_histograms(
        data_dict={"Members": target_member_feats[:, 1], "Non-members": target_nonmember_feats[:, 1]},
        xlabel="Per-sample cross-entropy loss",
        title="Loss Distribution: Members vs Non-members",
        log_y=True,
    )
    helper.plot_distribution_histograms(
        data_dict={"Members": mem_scores, "Non-members": nonm_scores},
        xlabel="Attack model membership score",
        title="Attack Model Score Distribution",
    )
else:
    print("Complete the TODO above first.")